<style>
    /* Style global du document */
    body {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        color: #2c3e50;
        line-height: 1.6;
        text-align: justify;
    }
    
    /* Style des titres */
    h1 {
        color: #1a5276;
        border-bottom: 3px solid #2980b9;
        padding-bottom: 8px;
        margin-top: 35px;
        font-size: 26px;
        text-align: center;
    }
    h2 {
        color: #2980b9;
        border-bottom: 1px solid #d4ac0d;
        padding-bottom: 5px;
        margin-top: 25px;
        font-size: 20px;
    }
    h3 {
        color: #117864;
        font-size: 16px;
    }

    /* Style des encadrés d'informations / KPI */
    .kpi-card {
        background-color: #f4f6f7;
        border-left: 5px solid #2980b9;
        padding: 15px;
        margin: 15px 0;
        border-radius: 4px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.05);
    }

    /* Style des tableaux */
    table.dataframe {
        border-collapse: collapse !important;
        width: 100% !important;
        margin: 20px 0 !important;
    }
    table.dataframe th {
        background-color: #1a5276 !important;
        color: white !important;
        text-align: left !important;
        padding: 10px !important;
    }
    table.dataframe td {
        padding: 8px !important;
        border-bottom: 1px solid #ddd !important;
    }
</style>
# HealthConnect Rwanda: Week 8 Final Model and Data Science Package

> **Track:** Data Science  
> **Client:** HealthConnect Rwanda  
> **Author:** Dorian NJAMI  
> **Programme:** AnalystLab Africa Experience Lab  
> **Project phase:** Week 8 - Final Integration, Documentation and Presentation

## Project Overview

This notebook continues the Week 6 and Week 7 work. It summarizes the Week 7 model tests, incorporates relevant Data Analytics findings, checks the admission-target lineage in the supplied workbook, and records the final integration decision.

The project workbook `HC_Clinical_Transactions.xlsx` contains an `Admissions` worksheet. The Week 6 master creates `admission_indicator` by marking an appointment positive when its ID matches an ID in that sheet. The source audit finds 586 matching admission rows and 4,414 admission rows whose appointment IDs do not match the appointment table. The admission source exists in this project package, while the unmatched keys still need investigation.

## Notebook Structure

### Part I - Week 7 to Week 8 Transition
1. Summarize the Week 7 results, refinements and remaining issues
2. Include Data Analytics findings that affect feature and business decisions

### Part II - Week 8 Final Integration and Model Readiness
1. Audit the target against source records and explain the unmatched keys
2. Set prediction-time and feature requirements
3. Confirm the final model decision and give evidence-based recommendations for weak performance
4. Document business suitability, limitations, cross-track contribution and ML Engineering requirements
5. Provide a concise model summary and presentation-ready findings

This notebook documents an analytical prototype and a readiness decision. It does not authorize clinical use or claim no-show prediction capability.

## Part I - Week 7 to Week 8 Transition

The Week 8 brief asks for a short transition from the previous work, without repeating the full Week 7 report.

| Transition item | Week 7 to Week 8 summary |
|---|---|
| Main Week 7 output | Tested admission-classification workflow comparing a prior-probability reference, Week 6-style Logistic Regression, refined Logistic Regression and Random Forest. |
| Main test result | On the Nov-Dec holdout, refined Logistic Regression AP was 0.124 and ROC-AUC 0.466; prior reference AP was 0.134 and ROC-AUC 0.500. |
| Major issue | The candidates did not beat the simple reference. The label is generated from linked Admissions rows, but 4,414 source keys are unmatched and may affect label completeness. |
| Refinements | Removed the direct target copy and constant feature, retained appointment rows, added strictly prior appointment history, used chronological splits and compared two model types. |
| Component ready | The model testing and evidence package is ready for review. No predictive model is approved for operational use. |
| Still to integrate | Investigate the 4,414 unmatched admission keys; confirm target meaning and feature timestamps; then rebuild and retest if the data owner validates the label. |
| Tracks involved | Data Analytics and the data owner for source/key review; Data Science for label/features and retest; ML Engineering for a future model handoff; Project Management for final readiness. |
| Week 8 demonstration | Show the Week 7 comparison, the source-key audit, the no-go decision and the evidence needed before model selection can resume. |

**Interpretation.** Week 7 did not demonstrate model improvement. The local Admissions worksheet supplies the target source, but the unmatched keys require review before negative labels and performance errors can be trusted fully.

### Week 7 Results Retained for Traceability

These scores came from the Week 7 notebook and use the label then available in the cleaned master. They describe performance against the linked-record proxy, not yet a source-owner-validated admission outcome.

| Candidate | Average precision | ROC-AUC | Precision at 0.50 | Recall at 0.50 | F1 at 0.50 | Balanced accuracy |
|---|---:|---:|---:|---:|---:|---:|
| Prior-probability reference | 0.134 | 0.500 | 0.000 | 0.000 | 0.000 | 0.500 |
| Week 6-style Logistic Regression | 0.131 | 0.467 | 0.120 | 0.473 | 0.192 | 0.469 |
| Refined Logistic Regression | 0.124 | 0.466 | 0.120 | 0.445 | 0.190 | 0.471 |
| Refined Random Forest | 0.138 | 0.493 | 0.103 | 0.055 | 0.071 | 0.491 |

Average precision (AP) is the primary ranking metric for the imbalanced positive class; ROC-AUC is complementary, while precision, recall, F1 and balanced accuracy depend on the stated 0.50 threshold. The paired bootstrap interval for refined Logistic Regression's AP difference versus the prior reference was -0.023 to 0.017 and included zero. No meaningful improvement was demonstrated. The 4,414 unmatched admission keys are an additional reason to qualify the Week 7 metrics.

### Data Analytics Findings Relevant to Data Science

The validation document provides these cross-track findings. They are attributed to Data Analytics; this notebook does not independently recalculate every dashboard test.

| Finding | Data Science implication |
|---|---|
| KPI reconciliation and date-slicer checks passed. | Supports the reported dashboard measures, not the admission-model target. |
| Matched-bill amounts were similar to unmatched bills (KS p=0.66); status and severity shares differed by under one percentage point. | Billing findings can be generalized cautiously with the matched-subset caveat; billing is not added to this admission model. |
| Matched monthly billing trend slope was -0.3M/month (p=0.37), with 5.9% variation. | No sustained billing trend was demonstrated; this is not an attendance trend. |
| Facility rating tests p=0.79/0.80 and department rating p=0.29 were not significant. | Exclude rating measures. `departmentname` is a category, not a rating. |
| Revenue Summary 1.13bn did not reconcile to 9.31bn billing and was removed from the dashboard. | Do not use this summary as a model feature or reconciled KPI. |
| No attendance/no-show field was found. | No missed-appointment model can be trained from the supplied outcome fields. |
| Gender/chronic condition considered usable; age, prior appointment count, visit type, department, month, weekday usable with caveats; diagnosis/severity held for timing review. | Carry these as provisional feature decisions and verify availability at prediction time. Prior-admission history needs key validation. |

Feedback linkage was 1,756 of 21,064 responses (about 8%). Linked/unlinked complaint rates were 40.9%/40.4%; mean ratings 2.93/2.99 (p=0.05). This may support a broader project insight with the linkage caveat, but feedback is not a predictor here.

## Part II - Week 8 Final Integration and Model Readiness

### 2.1 - Admission Target and Source-Key Audit

The `Admissions` worksheet is present and was used to construct `admission_indicator`. The audit below reproduces the join and quantifies unmatched keys. A complete binary field is not enough by itself: source relationships also need to be credible.

In [1]:
from pathlib import Path

import pandas as pd

# Locate the project root whether Jupyter starts here or in the Notebook folder.
possible_roots = [Path.cwd(), Path.cwd().parent]
project_root = next((root for root in possible_roots
                     if (root / "Data/Cleaned_Datasets/healthconnect_master_clean.csv").exists()), None)
if project_root is None:
    raise FileNotFoundError("Could not locate the project's cleaned master dataset.")

master_path = project_root / "Data/Cleaned_Datasets/healthconnect_master_clean.csv"
source_path = project_root / "Data/Original_Datasets/HC_Clinical_Transactions.xlsx"
master = pd.read_csv(master_path, low_memory=False)
appointments = pd.read_excel(source_path, sheet_name="Appointments")
admissions = pd.read_excel(source_path, sheet_name="Admissions")

# Compare appointment keys using trimmed string values.
appointment_keys = set(appointments["AppointmentID"].dropna().astype("string").str.strip())
admission_keys = admissions["AppointmentID"].dropna().astype("string").str.strip()
matched = admission_keys.isin(appointment_keys)
linked_keys = set(admission_keys.loc[matched])

# Recreate the Week 6 rule: a matched admission row makes an appointment positive.
source_label = master["appointmentid"].astype("string").str.strip().isin(linked_keys)
master_label = pd.to_numeric(master["admission_indicator"], errors="coerce")

lineage_audit = pd.DataFrame([
    {"Check": "Appointment rows", "Observed": len(appointments)},
    {"Check": "Admissions worksheet rows", "Observed": len(admissions)},
    {"Check": "Unique AdmissionID values", "Observed": admissions["AdmissionID"].nunique()},
    {"Check": "Distinct Admission AppointmentID values",
     "Observed": admissions["AppointmentID"].nunique()},
    {"Check": "Duplicate Admission AppointmentID rows",
     "Observed": int(admissions["AppointmentID"].duplicated().sum())},
    {"Check": "Admissions rows linked to an appointment", "Observed": int(matched.sum())},
    {"Check": "Admissions rows with unmatched AppointmentID", "Observed": int((~matched).sum())},
    {"Check": "Distinct linked appointment IDs", "Observed": len(linked_keys)},
    {"Check": "Positive labels in cleaned master", "Observed": int(master_label.sum())},
    {"Check": "Master label equals recreated link flag for every row",
     "Observed": bool(master_label.eq(source_label.astype("int8")).all())},
])
print("Source-lineage audit:\n" + lineage_audit.to_string(index=False))

Source-lineage audit:
                                                Check Observed
                                     Appointment rows     5000
                            Admissions worksheet rows     5000
                            Unique AdmissionID values     5000
              Distinct Admission AppointmentID values     5000
               Duplicate Admission AppointmentID rows        0
             Admissions rows linked to an appointment      586
         Admissions rows with unmatched AppointmentID     4414
                      Distinct linked appointment IDs      586
                    Positive labels in cleaned master      586
Master label equals recreated link flag for every row     True


**Interpretation of the source-key audit.** The repository workbook contains 5,000 Admissions rows. Of these, 586 `AppointmentID` values match the 5,000 appointment records and 4,414 do not. The cleaned master has 586 positive labels and reproduces the Week 6 rule: a matching admission row sets the appointment label to 1.

This confirms that the Admission worksheet exists and explains how the label was constructed. It does not explain why most source IDs are unmatched. Those records could reflect a key-format, source-scope or referential-integrity problem; no automatic ID repair should be applied. The 0 labels currently mean “no linked Admissions row found,” not necessarily a confirmed clinical non-admission. The separate finding that no attendance/no-show field is available remains applicable.

### 2.2 - Prediction Time and Feature Decisions

The proposed prediction point is **after the initial clinical assessment and before the admission decision/outcome**, once each selected field has actually been recorded. This is a design proposal that requires confirmation from the clinical/data owner. A booking-time model cannot use a diagnosis or severity value recorded later; an arrival-time model needs its own timestamped feature set.

| Feature | Week 8 decision | Reason |
|---|---|---|
| Gender, chronic condition | Potentially usable | Data Analytics reports these as usable; confirm definitions and missingness. |
| Age, prior appointment count, visit type, department, month and weekday | Potentially usable with caveats | Confirm availability before prediction and define history windows. |
| Diagnosis and severity | Hold | Same-visit timing is unclear; use only if timestamps show they precede the decision. |
| Prior admission count/rate | Hold | The Admissions sheet exists, but 4,414 unmatched source keys may make this history incomplete. |
| Facility/department ratings | Exclude | Data Analytics found no significant differences; department name itself is a separate category. |
| Billing, payments, revenue, complaint/feedback | Exclude from this model | Different outcome and potentially observed after prediction; revenue summary failed reconciliation. |
| `admissions_record_count` | Prohibit as a predictor | It duplicates the existing label and leaks it directly. |
| Attendance/no-show outcome | Unavailable | The validation summary reports no such field; new validated outcome data are required for attendance prediction. |

The Week 7 feature set is not approved unchanged. First review the unmatched admission keys and target semantics; then retain only confirmed pre-outcome variables and rerun temporal validation.

### 2.3 - Final Candidate Selection

**No predictive model is approved as the final operational candidate.** Week 7 Logistic Regression did not beat the prior reference on average precision or ROC-AUC. The Random Forest showed severe overfitting, with training AP/ROC-AUC of 0.815/0.969 versus validation values of 0.117/0.499. These results provide no evidence that deploying the current model would improve decisions.

The target source does exist in the repository, but 4,414 of 5,000 Admissions keys do not match an appointment. Until the source owner explains this relationship and confirms the outcome meaning, model errors and negative labels remain qualified. The Week 8 Data Science outcome is a documented **no-go decision for predictive use**, a traceable comparison, and an evidence-based plan to address weak performance and data quality.

### 2.4 - Final Integration Readiness and HC-POD Record

| Readiness item | Current response | Status |
|---|---|---|
| Final component | Week 7 model-test evidence plus Week 8 source-key and readiness review | Evidence package ready to present |
| Problem addressed | Whether current data support reliable admission prediction for the broader appointment-support goal | No-show objective remains unsupported |
| Current model status | No final candidate approved; Week 7 metrics use a linked-record proxy | Not ready for predictive use |
| Week 7 improvement | Leakage checks, chronological evaluation, candidate comparison and error analysis | Completed; no gain demonstrated |
| Dependency on Data Analytics/data owner | Review the 4,414 unmatched Admission IDs, confirm the proper join and meaning of unmatched rows, and verify target semantics/timestamps | Pending |
| Output for ML Engineering | If a model later passes validation: versioned feature schema, prediction time, fitted pipeline, input/output contract and validation summary | No model artifact to deploy now |
| Readiness evidence | Week 7 outputs, executed source-key audit, Analytics validation summary and issue register | Review package ready; model integration not ready |
| Remaining limitations | 4,414 unmatched admission keys, no no-show label, diagnosis/severity timing, one-year history, weak model ranking | Open |

#### Cross-Track Integration Record

| Required item | Week 8 record |
|---|---|
| Track collaborated with | Data Analytics (Agu Precious); ML Engineering (Charles; repository and handoff information shared, feedback pending as of 2026-11-06); Project Management for final readiness |
| Output received | Data Analytics validation summary with dashboard, billing, rating and target-inventory findings |
| Output to provide | Week 7 model results and this audit: 5,000 Admissions rows, 586 matching keys, 4,414 unmatched keys, and the Week 6 label rule. Repository and Week 7 summary shared with Charles (ML Engineering); response/review pending as of 2026-11-06. GitHub: https://github.com/Dorsin/healthconnect-rwanda |
| Final integration activity | Information shared with Charles; ML Engineering review is pending. With Data Analytics/data owner, inspect unmatched Admission IDs, confirm target meaning and prediction time, then decide whether to retest. |
| What changed | The local target source is confirmed present; unmatched keys are the specific data issue. The Week 7 candidate remains provisional and prior-admission features are held. Charles has the repository and handoff summary; no engineering feedback has been received yet. |
| How this improves the solution | Prevents broken or incomplete links from distorting model labels and downstream performance claims. |
| Evidence | Data Analytics validation summary, executed audit in Section 2.1, and the user-reported handoff to Charles; retain the message exchange and any later engineering/source-owner response. |
| Retest | Pending explanation/correction of unmatched keys; repeat temporal evaluation if the validated target changes. |

The Data Analytics findings and ML Engineering handoff are documented. Charles has not yet returned feedback; joint key reconciliation and model retesting remain pending.

### 2.5 - Information to Provide to ML Engineering

The repository and a Week 7/Week 8 model summary have been shared with Charles in the ML Engineering track; no response has been received as of 2026-11-06. The handoff is a **no-deployment notice and future requirements checklist**. There is no model artifact to deploy because the tested candidates have not shown useful performance and source-key quality needs review. Repository: https://github.com/Dorsin/healthconnect-rwanda

| Deployment information | Current status / requirement |
|---|---|
| Target definition | Week 6 used `admission_indicator = 1` when an Admissions `AppointmentID` matched an appointment; validate unmatched rows and target semantics with the data owner. |
| Prediction point | Proposed after initial clinical assessment and before the admission decision; confirm actual workflow and timestamps. |
| Final model/artifact | None approved. Do not integrate the Week 7 candidate as a decision service. |
| Feature schema | Provisional; include only fields confirmed to exist before prediction. Hold admission-history, diagnosis and severity fields pending evidence. |
| Input/output behavior | If a future model is approved, document required types, missing values, unseen categories, probability meaning, decision threshold and error behavior. |
| Dependencies/configuration | Pin package versions, record configuration, source-data version, model version and reproducible build steps. |
| Testing and monitoring | Add schema validation, unit/integration tests, logging, drift/performance monitoring and rollback instructions before any operational use. |
| Use restrictions | No no-show prediction, clinical recommendation or automated scheduling decision is supported by current evidence. |

When a valid model exists, Data Science should send the serialized pipeline, exact input schema, preprocessing, output definition, threshold rationale, validation results by time/segment, known risks and version details. ML Engineering should not compensate for weak performance by masking it in the pipeline.

### 2.6 - Business Suitability and Non-Technical Model Summary

**What the Week 7 results say:** on the 821-row final holdout, the admission-link label prevalence was 13.4%. The prior reference had AP 0.134 and ROC-AUC 0.500. The selected refined Logistic Regression had AP 0.124 and ROC-AUC 0.466. At threshold 0.50, its proxy-label confusion matrix was 353 true negatives, 358 false positives, 61 false negatives and 49 true positives. Only 49 of 407 positive alerts were correct (about 12% precision), while recall was 44.5%. These are errors against the current linked-record label and may change after unmatched keys are resolved.

**Strengths:** Week 7 used a time-ordered holdout, compared candidates with a simple baseline, kept preprocessing inside model pipelines, checked a direct target copy and documented uncertainty. These are useful evaluation practices.

**Weaknesses:** The selected candidate ranked below the prior reference; the Random Forest overfit strongly; the proxy target has 4,414 unmatched admission keys; diagnosis/severity timing remains unconfirmed; and no attendance/no-show label exists. Current metrics do not support operational use.

**Business decision:** do not use the current model scores to prioritize patients, change care, or automate appointment actions. Continue the organization's approved human-led workflow while data and model issues are addressed. No-show prediction requires a separately defined and validated attendance outcome.

### 2.7 - Recommendations for Weak Model Performance

The current results call for a controlled improvement plan. Moving to more complex algorithms or lowering the threshold will not fix an unreliable label or create predictive signal.

1. **Pause operational use.** The refined model AP (0.124) is below test prevalence (0.134) and ROC-AUC is 0.466. Random Forest AP (0.138) is only slightly above prevalence, ROC-AUC is 0.493, and recall is 5.5% at 0.50. Neither candidate shows reliable ranking.
2. **Investigate the target links.** Have the data owner review the 4,414 unmatched `AppointmentID` values. Check identifier format, table grain, duplicate rules, source scope and whether records belong to another appointment extract. Do not fuzzy-match or relabel them without an authoritative mapping.
3. **Align target and business objective.** Admission prediction and missed-appointment prediction are different tasks. If attendance is the goal, collect a validated attended/no-show outcome with clear definitions and event dates.
4. **Confirm prediction time and feature availability.** Exclude diagnosis/severity until timestamps prove they precede the decision. Hold prior-admission features until history links are validated. Exclude post-outcome billing, feedback and admission counts.
5. **Rebuild simple benchmarks after validation.** Start with a prevalence reference and regularized Logistic Regression. Add more models only after target and feature quality are sound. Fit imputation, encoding, scaling, class weights or resampling on training data only.
6. **Use future-oriented validation.** Keep a final chronological holdout untouched; with more years, add rolling-origin validation. Choose thresholds on validation data against agreed alert capacity and error costs. Report average precision/PR curve, ROC-AUC, precision, recall, F1, calibration and confusion counts, with uncertainty intervals and segment checks where sample sizes permit.
7. **Set a stakeholder acceptance rule.** Agree on minimum improvement over the reference, acceptable alert volume, recall and calibration. If no model meets the rule, record no-go and request better labels/features instead of deployment.
8. **Retest changes on fresh evidence.** After a justified source or feature correction, rerun the same temporal evaluation and compare on an untouched holdout. Record source version, target logic, feature list and before/after results.

#### Model Findings for the Final Presentation

- **Contribution:** tested a time-aware admission-classification workflow and reviewed its errors.
- **Evidence:** refined Logistic Regression AP 0.124/AUC 0.466 versus prior-reference AP 0.134/AUC 0.500; Random Forest showed severe overfitting.
- **Decision:** no candidate is approved for operational use; the results show no reliable improvement.
- **Data issue:** 586 of 5,000 Admissions keys link to appointments; 4,414 remain unmatched and need source-owner review.
- **Business limit:** the available target is admission linkage, not missed-appointment attendance; no-show prediction is unsupported.
- **Next step:** validate target links and pre-decision features, then rebuild and retest before ML Engineering integration.

### Week 8 Task and Output Coverage

| Week 8 task/output | Notebook evidence | Status |
|---|---|---|
| 1. Review Week 7 testing | Part I transition and results | Complete; label caveat included |
| 2. Confirm final candidate | Section 2.3 | No candidate approved |
| 3. Compare against baseline | Week 7 results table | Complete; same test rows, provisional labels |
| 4. Present relevant metrics | AP, ROC-AUC, recall, prevalence and bootstrap interval | Complete |
| 5. Explain strengths and weaknesses | Sections 2.3, 2.6 and 2.7 | Complete |
| 6. Document modeling decisions | Feature decisions and no-go decision | Complete |
| 7. Explain business suitability | Section 2.6 | Not suitable for operational use |
| 8. Incorporate Data Analytics findings | Part I analytics summary and source-sheet clarification | Complete |
| 9. Communicate requirements to ML Engineering | Section 2.5 | Repository and no-deployment summary shared with Charles; feedback pending as of 2026-11-06 |
| 10. Document limitations and risks | Sections 2.1-2.6 | Complete; unmatched-key review open |
| 11. State permitted and prohibited uses | Section 2.6 | No clinical, automated scheduling or no-show use |
| 12. Prepare model content for presentation | Section 2.7 | Slide-ready model findings included |
| 13. Prepare non-technical model summary | Section 2.6 | Complete |
| 14. Organize a professional notebook/report | Numbered sections, interpretations and evidence tables | Complete |
| Final candidate model | Sections 2.3 and 2.5 | None approved or deployable |
| Evaluation and baseline comparison | Week 7 results table | Included; based on linked-record proxy |
| Error analysis summary | Section 2.6 confusion counts | Included; errors are proxy-label errors and require caution |
| Feature and model decisions | Sections 2.2-2.5 | Included |
| Cross-track contribution | Section 2.4 | Analytics findings recorded; ML Engineering handoff shared with Charles; response, key review and retest pending |
| Model documentation and presentation material | Sections 2.5 and 2.7 | Included |

## Week 8 Data Science Project Summary

| Required summary item | Current record |
|---|---|
| Final Data Science component | Week 7 model-test evidence and Week 8 target-link/readiness decision |
| Week 7 result | Refined Logistic Regression AP 0.124/AUC 0.466; prior reference AP 0.134/AUC 0.500 on the linked-record label |
| Most important new finding | The local Admissions worksheet has 5,000 rows; 586 keys match appointments and 4,414 do not. The source exists, but the unmatched keys need investigation. |
| Final candidate | None approved because results are weak and target-link quality is unresolved |
| Cross-track work | Data Analytics validation summary received; repository and model handoff shared with Charles; no response as of 2026-11-06; key review and retest pending |
| Business relevance | Prevents unreliable scores from entering a workflow; no-show prediction remains unsupported |
| Remaining limitations | Unmatched admission keys, target semantics, diagnosis/severity timing, one-year history, no attendance label |
| Required next action | Review unmatched keys with Data Analytics/data owner, confirm target and prediction time, rebuild and retest if evidence supports it |
| Readiness decision | Evidence package ready for presentation; predictive model and ML Engineering integration are not ready |

This package summarizes Week 7 without repeating its full report. It records the source worksheet that exists, the unresolved key-quality issue, and a defensible no-go decision for the current models.